In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy.io import loadmat
from sklearn.datasets import fetch_olivetti_faces
from sklearn.metrics import accuracy_score
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

ROLL_NUMBER = "p26cs0006"
SEED = int("".join(c for c in ROLL_NUMBER if c.isdigit()))

np.random.seed(SEED)
tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)

REPEATS = 30
EPOCHS = 3
BAG_SIZE = 5
BOOST_ROUNDS = 5
BATCH_SIZE = 32

try:
    data = fetch_olivetti_faces()
    images = data.images.astype("float32")
    y = data.target
except Exception:
    if not os.path.exists("olivettifaces.mat"):
        raise FileNotFoundError("Place olivettifaces.mat in the notebook folder.")
    faces = loadmat("olivettifaces.mat")["faces"].T.astype("float32")
    faces = (faces - faces.min()) / (faces.max() - faces.min())
    images = faces.reshape(400, 64, 64).transpose(0, 2, 1)
    y = np.arange(400) // 10

images = images[..., np.newaxis]
classes = np.unique(y)
K = len(classes)

train_idx = []
val_idx = []
test_idx = []

for c in classes:
    idx = rng.permutation(np.where(y == c)[0])
    train_idx.extend(idx[:6])
    val_idx.extend(idx[6:8])
    test_idx.extend(idx[8:10])

train_idx = np.array(train_idx)
val_idx = np.array(val_idx)
test_idx = np.array(test_idx)

X_train, y_train = images[train_idx], y[train_idx]
X_val, y_val = images[val_idx], y[val_idx]
X_test, y_test = images[test_idx], y[test_idx]

X_masked = X_test.copy()
X_masked[:, 24:40, 24:40, :] = 0

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Classes:", K)


def make_cnn(depth=2):
    model = Sequential()
    model.add(Input(shape=(64, 64, 1)))

    filters = [16, 32, 64]

    for i in range(depth):
        model.add(Conv2D(filters[i], 3, activation="relu", padding="same"))
        model.add(MaxPooling2D(2))

    model.add(Flatten())
    model.add(Dense(64, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(K, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


def train_cnn(X, y, depth=2, weights=None):
    model = make_cnn(depth)

    model.fit(
        X,
        y,
        sample_weight=weights,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    return model


def sample_training_set():
    selected = []

    for c in classes:
        idx = np.where(y_train == c)[0]
        selected.extend(rng.choice(idx, size=6, replace=True))

    selected = np.array(selected)
    return X_train[selected], y_train[selected]


def fit_bagging(X, y, depth=2, members=BAG_SIZE):
    models = []

    for _ in range(members):
        idx = rng.integers(0, len(y), size=len(y))
        model = train_cnn(X[idx], y[idx], depth)
        models.append(model)

    return models


def bag_probabilities(models, X):
    predictions = [model.predict(X, verbose=0) for model in models]
    return np.mean(predictions, axis=0)


def fit_boosting(X, y, depth=2, rounds=BOOST_ROUNDS):
    models = []
    alphas = []
    weights = np.ones(len(y)) / len(y)

    for _ in range(rounds):
        model = train_cnn(X, y, depth, weights)
        probabilities = model.predict(X, verbose=0)
        pred = classes[np.argmax(probabilities, axis=1)]

        error = np.sum(weights * (pred != y))

        if error >= 1 - 1 / K:
            break

        alpha = np.log(
            (1 - min(error, 1 - 1e-12)) / max(error, 1e-12)
        ) + np.log(K - 1)

        models.append(model)
        alphas.append(alpha)

        weights *= np.exp(alpha * (pred != y))
        weights /= weights.sum()

        if error < 1e-12:
            break

    return models, np.array(alphas)


def boost_probabilities(models, alphas, X):
    scores = np.zeros((len(X), K))

    for model, alpha in zip(models, alphas):
        pred = np.argmax(model.predict(X, verbose=0), axis=1)
        scores[np.arange(len(X)), pred] += alpha

    total = alphas.sum()

    if total == 0:
        return np.ones((len(X), K)) / K

    return scores / total


def bias_variance(probabilities, y_true):
    true_vectors = np.eye(K)[y_true]
    mean_probabilities = probabilities.mean(axis=0)

    bias_squared = np.mean(
        np.sum((mean_probabilities - true_vectors) ** 2, axis=1)
    )

    variance = np.mean(
        np.sum(
            (probabilities - mean_probabilities[None, :, :]) ** 2,
            axis=2
        )
    )

    return bias_squared, variance


single_predictions = []
bag_predictions = []
boost_predictions = []

single_accuracy = []
bag_accuracy = []
boost_accuracy = []

for r in range(REPEATS):
    print("Resampling run:", r + 1, "of", REPEATS)

    X_r, y_r = sample_training_set()

    single = train_cnn(X_r, y_r, depth=2)
    single_prob = single.predict(X_test, verbose=0)

    bag = fit_bagging(X_r, y_r, depth=2)
    bag_prob = bag_probabilities(bag, X_test)

    boosted, alphas = fit_boosting(X_r, y_r, depth=2)
    boost_prob = boost_probabilities(boosted, alphas, X_test)

    single_predictions.append(single_prob)
    bag_predictions.append(bag_prob)
    boost_predictions.append(boost_prob)

    single_accuracy.append(
        accuracy_score(y_test, np.argmax(single_prob, axis=1))
    )
    bag_accuracy.append(
        accuracy_score(y_test, np.argmax(bag_prob, axis=1))
    )
    boost_accuracy.append(
        accuracy_score(y_test, np.argmax(boost_prob, axis=1))
    )

    del single, bag, boosted
    tf.keras.backend.clear_session()

single_predictions = np.array(single_predictions)
bag_predictions = np.array(bag_predictions)
boost_predictions = np.array(boost_predictions)

bias_rows = []

for name, predictions, accuracies in [
    ("Single CNN", single_predictions, single_accuracy),
    ("CNN Bagging", bag_predictions, bag_accuracy),
    ("CNN Boosting", boost_predictions, boost_accuracy)
]:
    bias, variance = bias_variance(predictions, y_test)

    bias_rows.append({
        "Model": name,
        "Bias Squared": bias,
        "Variance": variance,
        "Mean Test Accuracy": np.mean(accuracies),
        "Accuracy Std": np.std(accuracies, ddof=1)
    })

bias_table = pd.DataFrame(bias_rows)

print("\nPART A: BIAS AND VARIANCE")
print(bias_table.to_string(index=False))


depth_results = []

for depth in [1, 2, 3]:
    print("Testing CNN depth:", depth)

    single = train_cnn(X_train, y_train, depth)
    single_prob = single.predict(X_test, verbose=0)

    bag = fit_bagging(X_train, y_train, depth, members=100)
    bag_prob = bag_probabilities(bag, X_test)

    depth_results.append({
        "CNN Blocks": depth,
        "Single CNN Accuracy": accuracy_score(
            y_test, np.argmax(single_prob, axis=1)
        ),
        "Bagging Accuracy": accuracy_score(
            y_test, np.argmax(bag_prob, axis=1)
        ),
        "Single CNN Bias Squared": bias_variance(
            np.array([single_prob]), y_test
        )[0],
        "Single CNN Variance": bias_variance(
            np.array([single_prob]), y_test
        )[1]
    })

    del single, bag
    tf.keras.backend.clear_session()

depth_table = pd.DataFrame(depth_results)

print("\nPART B: CNN ARCHITECTURE COMPARISON")
print(depth_table.to_string(index=False))


selected_depth = 2

single = train_cnn(X_train, y_train, selected_depth)
bag = fit_bagging(X_train, y_train, selected_depth, members=100)
boosted, alphas = fit_boosting(
    X_train, y_train, selected_depth, rounds=BOOST_ROUNDS
)

single_clean = single.predict(X_test, verbose=0)
bag_clean = bag_probabilities(bag, X_test)
boost_clean = boost_probabilities(boosted, alphas, X_test)

single_masked = single.predict(X_masked, verbose=0)
bag_masked = bag_probabilities(bag, X_masked)
boost_masked = boost_probabilities(boosted, alphas, X_masked)

final_rows = []

for split, truth, predictions in [
    ("Clean test", y_test, [
        ("Single CNN", single_clean),
        ("CNN Bagging", bag_clean),
        ("CNN Boosting", boost_clean)
    ]),
    ("Masked test", y_test, [
        ("Single CNN", single_masked),
        ("CNN Bagging", bag_masked),
        ("CNN Boosting", boost_masked)
    ])
]:
    for name, probabilities in predictions:
        final_rows.append({
            "Test Set": split,
            "Model": name,
            "Accuracy": accuracy_score(
                truth, np.argmax(probabilities, axis=1)
            ),
            "Mean Winning Probability": probabilities.max(axis=1).mean()
        })

final_table = pd.DataFrame(final_rows)

print("\nPART D: CLEAN AND MASKED TEST")
print(final_table.to_string(index=False))


different = np.where(
    np.argmax(bag_clean, axis=1) !=
    np.argmax(boost_clean, axis=1)
)[0]

print("\nBAGGING AND BOOSTING DIFFERENCE")

if len(different) == 0:
    print("No prediction difference on the clean test set.")
else:
    i = different[0]

    print("True identity:", y_test[i])
    print("Bagging prediction:", np.argmax(bag_clean[i]))
    print("Boosting prediction:", np.argmax(boost_clean[i]))

    plt.imshow(X_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    plt.show()

plt.figure(figsize=(8, 4))
plt.bar(
    bias_table["Model"],
    bias_table["Variance"]
)
plt.ylabel("Measured variance")
plt.title("Variance Comparison")
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(
    bias_table["Model"],
    bias_table["Bias Squared"]
)
plt.ylabel("Measured bias squared")
plt.title("Bias Comparison")
plt.show()


downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data
Training: (240, 64, 64, 1)
Validation: (80, 64, 64, 1)
Test: (80, 64, 64, 1)
Classes: 40
Resampling run: 1 of 30


Resampling run: 2 of 30
Resampling run: 3 of 30
Resampling run: 4 of 30
Resampling run: 5 of 30
Resampling run: 6 of 30
Resampling run: 7 of 30
Resampling run: 8 of 30
Resampling run: 9 of 30
Resampling run: 10 of 30
Resampling run: 11 of 30
Resampling run: 12 of 30
Resampling run: 13 of 30
Resampling run: 14 of 30
Resampling run: 15 of 30
Resampling run: 16 of 30
Resampling run: 17 of 30
Resampling run: 18 of 30
Resampling run: 19 of 30
Resampling run: 20 of 30
Resampling run: 21 of 30
Resampling run: 22 of 30
Resampling run: 23 of 30
Resampling run: 24 of 30
Resampling run: 25 of 30
Resampling run: 26 of 30
Resampling run: 27 of 30
Resampling run: 28 of 30
Resampling run: 29 of 30
Resampling run: 30 of 30

PART A: BIAS AND VARIANCE
       Model  Bias Squared  Variance  Mean Test Accuracy  Accuracy Std
  Single CNN      0.973817  0.000107            0.055833      0.028755
 CNN Bagging      0.973055  0.000212            0.072500      0.045887
CNN Boosting      0.939596  0.333854       